In [22]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import datetime
import matplotlib.pyplot as plt
import numpy as np
import sumo_rl
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from sumo_rl.environment.observations import ObservationFunction
from gymnasium import spaces
from pathlib import Path

In [23]:
# Hyperparameters
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.95
batch_size = 64
learning_rate = 1e-3
num_seconds=1000
episodes = 50

In [24]:
def priority_reward_fn(ts):
    """
    Custom reward function that heavily penalizes stopped ambulances and buses.
    ts: The TrafficSignal object provided by sumo_rl.
    """
    reward = 0
    # Access the TraCI connection directly through the traffic signal
    traci = ts.sumo 
    
    # Iterate through all incoming lanes controlled by this specific traffic light
    for lane in ts.lanes:
        # Get the IDs of all vehicles currently on this lane
        vehicles = traci.lane.getLastStepVehicleIDs(lane)
        
        for veh in vehicles:
            # Get the vehicle's current speed
            speed = traci.vehicle.getSpeed(veh)
            
            # If the vehicle is practically stopped (waiting at a red light)
            if speed < 0.1:
                # Retrieve the vehicle type (defined in our .rou.xml file)
                v_type = traci.vehicle.getTypeID(veh)
                
                # Apply custom penalty weights
                if v_type == "ambulance":
                    reward -= 50.0  # Massive penalty for delaying an ambulance
                elif v_type == "bus":
                    reward -= 5.0   # Moderate penalty for delaying a bus
                else:
                    reward -= 1.0   # Standard penalty for normal cars
                    
    return reward

In [25]:
class PriorityObservationFunction(ObservationFunction):
    def __init__(self, ts):
        super().__init__(ts)
        # We need to define the shape of our observation array.
        # Let's say: [density_N, density_S, density_E, density_W, emergency_present_N, emergency_present_S...]
        pass 

    def __call__(self):

        traci = self.ts.sumo
        obs = []
        
        # normal traffic density
        density = self.ts.get_lanes_density()
        obs.extend(density)
        
        # whether an ambulance is waiting on each lane
        for lane in self.ts.lanes:
            emergency_waiting = 0.0
            vehicles = traci.lane.getLastStepVehicleIDs(lane)
            for veh in vehicles:
                if traci.vehicle.getTypeID(veh) == "ambulance":
                    emergency_waiting = 1.0
                    break
            obs.append(emergency_waiting)
            
        return np.array(obs, dtype=np.float32)

    def observation_space(self):
        return spaces.Box(low=0., high=1., shape=(len(self.ts.lanes) * 2,), dtype=np.float32)

In [26]:
# Initialize PettingZoo environment

base = Path("3x3_priority")

route_files = [
    base / "vtypes.add.xml",
    base / "cars.rou.xml",
    base / "buses.rou.xml",
    base / "bikes.rou.xml",
    base / "ambulance.rou.xml",
]

route = ",".join(str(p) for p in route_files)

env = sumo_rl.parallel_env(
    net_file="3x3_priority/3x3Grid2lanes.net.xml",
    route_file=route,
    reward_fn=priority_reward_fn,
    observation_class=PriorityObservationFunction,
    use_gui=False, # Keep False for training
    num_seconds=num_seconds,
    delta_time=5,
    sumo_warnings = False
)

In [27]:
# Get dimensions for the network
sample_agent = env.possible_agents[0]
obs_dim = env.observation_space(sample_agent).shape[0]
action_dim = env.action_space(sample_agent).n

In [28]:
class QLearningAgent:
    def __init__(self, action_dim):
        self.action_dim = action_dim
        self.q_table = {}  # Using a dict to handle large potential state spaces
        self.epsilon = epsilon
        self.lr = learning_rate
        self.gamma = gamma

    def get_state_key(self, observation):
        # Discretize: density into 3 bins (0: low, 1: med, 2: high) 
        # and keep emergency flag as 0 or 1.
        discretized = []
        for val in observation:
            if val < 0.33:
                discretized.append(0)
            elif val < 0.66:
                discretized.append(1)
            else:
                discretized.append(2)
        return tuple(discretized)

    def choose_action(self, state_key):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        
        # If state never seen, default to 0s
        q_values = self.q_table.get(state_key, np.zeros(self.action_dim))
        return np.argmax(q_values)

    def learn(self, state, action, reward, next_state, done):
        state_key = self.get_state_key(state)
        next_state_key = self.get_state_key(next_state)
        
        # Initialize rows if needed
        if state_key not in self.q_table:
            self.q_table[state_key] = np.zeros(self.action_dim)
        if next_state_key not in self.q_table:
            self.q_table[next_state_key] = np.zeros(self.action_dim)
            
        # Q-Learning Formula: Q(s,a) = Q(s,a) + lr * [R + gamma * max(Q(s',a')) - Q(s,a)]
        old_value = self.q_table[state_key][action]
        next_max = np.max(self.q_table[next_state_key])
        
        new_value = old_value + self.lr * (reward + self.gamma * next_max * (1 - done) - old_value)
        self.q_table[state_key][action] = new_value

In [29]:
# Setup TensorBoard
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = f"runs/q_learning_{current_time}"
writer = SummaryWriter(log_dir)

# Initialize agents
agents = {agent_id: QLearningAgent(action_dim) for agent_id in env.possible_agents}

print(f"Training started. View logs with: tensorboard --logdir={log_dir}")

# Progress Bar
for ep in tqdm(range(episodes), desc="Training Episodes"):
    observations, infos = env.reset()
    total_reward = 0
    step_count = 0
    
    while env.agents:
        actions = {
            agent_id: agents[agent_id].choose_action(
                agents[agent_id].get_state_key(observations[agent_id])
            ) for agent_id in env.agents
        }
        
        next_obs, rewards, terminations, truncations, infos = env.step(actions)
        
        episode_step_reward = 0
        for agent_id in rewards:
            agents[agent_id].learn(
                observations[agent_id], 
                actions[agent_id], 
                rewards[agent_id], 
                next_obs[agent_id], 
                terminations[agent_id]
            )
            episode_step_reward += rewards[agent_id]
            
            # Decay epsilon per agent
            if agents[agent_id].epsilon > epsilon_min:
                agents[agent_id].epsilon *= epsilon_decay
        
        total_reward += episode_step_reward
        observations = next_obs
        step_count += 1

    # Log metrics to TensorBoard
    writer.add_scalar("Reward/Total", total_reward, ep)
    writer.add_scalar("Reward/Average_Per_Step", total_reward / step_count if step_count > 0 else 0, ep)
    writer.add_scalar("Epsilon", agents[sample_agent].epsilon, ep)
    writer.add_scalar("Q-Table/Size", len(agents[sample_agent].q_table), ep)

writer.close()
env.close()
print("Training Complete.")

Training started. View logs with: tensorboard --logdir=runs/q_learning_20260410-015158


Training Episodes:   0%|          | 0/50 [00:00<?, ?it/s]

Training Episodes: 100%|██████████| 50/50 [43:02<00:00, 51.65s/it] 


Training Complete.


In [33]:
%tensorboard --logdir ./runs

Reusing TensorBoard on port 6007 (pid 24624), started 0:00:49 ago. (Use '!kill 24624' to kill it.)